In [ ]:
import torch
import torch.nn as nn

class EDM_Precond(nn.Module):
    def __init__(self, model, sigma_data=1.0):
        super().__init__()
        self.model = model  # Your 1D SongUNet
        self.sigma_data = sigma_data # Global std of your log-returns (usually ~1.0)

    def forward(self, x, sigma):
        # EDM Preconditioning Coefficients [cite: 5432-5435]
        c_skip = self.sigma_data**2 / (sigma**2 + self.sigma_data**2)
        c_out = sigma * self.sigma_data / (sigma**2 + self.sigma_data**2).sqrt()
        c_in = 1 / (sigma**2 + self.sigma_data**2).sqrt()
        c_noise = 0.25 * sigma.log()

        # Net prediction formula[cite: 5430, 5520]: 
        # D_theta(x; sigma) = c_skip*x + c_out*F_theta(c_in*x, c_noise)
        return c_skip * x + c_out * self.model(c_in * x, c_noise)

In [ ]:
def edm_loss_step(model_precond, x_0, sigma_data=1.0):
    # 1. Sample sigma from log-normal 
    rnd_normal = torch.randn([x_0.shape[0], 1, 1], device=x_0.device)
    sigma = (rnd_normal * 1.2 - 1.2).exp()
    
    # 2. Add noise to clean log-returns
    noise = torch.randn_like(x_0)
    x_t = x_0 + sigma * noise
    
    # 3. Predict clean data through preconditioning wrapper
    x_hat = model_precond(x_t, sigma)
    
    # 4. Compute weighted MSE [cite: 5414, 5420]
    weight = (sigma**2 + sigma_data**2) / (sigma * sigma_data)**2
    loss = (weight * (x_hat - x_0)**2).mean()
    
    return loss

In [ ]:
@torch.no_grad()
def heun_sampler(model_precond, shape, num_steps=35, sigma_min=0.002, sigma_max=80, rho=7):
    # 1. Generate discretization schedule
    step_indices = torch.arange(num_steps)
    t_steps = (sigma_max**(1/rho) + step_indices / (num_steps - 1) * (sigma_min**(1/rho) - sigma_max**(1/rho)))**rho
    t_steps = torch.cat([t_steps, torch.zeros(1)]) # Add zero endpoint

    # 2. Start from pure Gaussian noise [cite: 5595]
    x = torch.randn(shape) * t_steps[0]
    
    # 3. Iterative denoising loop
    for i in range(num_steps):
        t_cur = t_steps[i]
        t_next = t_steps[i+1]
        
        # Euler step direction [cite: 5597]
        d_cur = (x - model_precond(x, t_cur)) / t_cur
        x_next = x + (t_next - t_cur) * d_cur
        
        # 2nd order correction if not at the last step [cite: 5600]
        if t_next > 0:
            d_prime = (x_next - model_precond(x_next, t_next)) / t_next
            x = x + (t_next - t_cur) * (0.5 * d_cur + 0.5 * d_prime)
        else:
            x = x_next
            
    return x